# 17. 탐지에서 분할로

이 노트북은 `16_YOLO_영상_실습.ipynb` 다음 단계로, 객체 탐지(object detection)에서 이미지 분할(segmentation)로 넘어가는 개념적 다리를 만드는 것이 목표입니다.

2장에서는 모델이 `무엇이 어디에 있는가?`를 bounding box로 답했습니다. 3장에서는 질문이 더 정밀해집니다.

- 탐지: `객체를 감싸는 직사각형은 어디인가?`
- 분할: `객체가 차지하는 픽셀은 정확히 어디인가?`

즉, segmentation은 객체의 위치를 대략적인 박스가 아니라 **픽셀 단위 mask**로 표현하는 문제입니다.

이번 노트북의 목표는 다음과 같습니다.

- classification, detection, segmentation의 출력 차이를 비교합니다.
- bounding box 방식의 한계를 이해합니다.
- segmentation label이 왜 mask 형태인지 이해합니다.
- semantic segmentation과 instance segmentation으로 넘어갈 준비를 합니다.


## 17-1. 준비

이번 노트북은 실제 모델 학습 없이 간단한 도형과 배열을 사용해 detection과 segmentation의 차이를 시각적으로 확인합니다.


In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.unicode_minus'] = False


## 17-2. 세 문제의 출력은 어떻게 다를까?

이미지 한 장을 입력했을 때 classification, detection, segmentation은 서로 다른 형태의 답을 냅니다.

- classification: 이미지 전체에 대한 클래스 하나 또는 클래스 확률 벡터
- detection: 각 객체의 클래스와 bounding box
- segmentation: 각 픽셀의 클래스 또는 객체 mask

입력 이미지는 같아도, 문제 정의가 바뀌면 라벨과 모델 출력이 함께 바뀝니다.


In [ ]:
image_h, image_w = 120, 160
yy, xx = np.mgrid[:image_h, :image_w]

circle_mask = (xx - 70) ** 2 + (yy - 60) ** 2 <= 32 ** 2
image = np.ones((image_h, image_w, 3), dtype=float)
image[..., :] = [0.92, 0.96, 1.0]
image[circle_mask] = [0.95, 0.35, 0.30]

bbox = (38, 28, 102, 92)  # x1, y1, x2, y2

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(image)
axes[0].set_title('Classification\nlabel = object')

axes[1].imshow(image)
x1, y1, x2, y2 = bbox
axes[1].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='lime', linewidth=3))
axes[1].set_title('Detection\nclass + bounding box')

axes[2].imshow(circle_mask, cmap='gray')
axes[2].set_title('Segmentation\npixel mask')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()


## 17-3. Bounding box의 한계

Bounding box는 객체 위치를 빠르고 간단하게 표현할 수 있습니다. 하지만 직사각형이기 때문에 객체의 실제 모양을 정확히 표현하지 못합니다.

예를 들어 사람, 자동차, 도로, 의료 영상의 병변처럼 모양이 복잡한 대상은 박스 안에 배경 픽셀이 많이 포함될 수 있습니다. 이 경우 `어디쯤 있는가`는 알 수 있지만 `어디까지가 객체인가`는 알기 어렵습니다.


In [ ]:
box_mask = np.zeros_like(circle_mask)
box_mask[y1:y2, x1:x2] = True

object_pixels = circle_mask.sum()
box_pixels = box_mask.sum()
background_inside_box = box_pixels - object_pixels

print(f'객체 픽셀 수: {object_pixels}')
print(f'박스 내부 픽셀 수: {box_pixels}')
print(f'박스 안에 함께 포함된 배경 픽셀 수: {background_inside_box}')
print(f'박스 내부 중 실제 객체 비율: {object_pixels / box_pixels:.2%}')


위 예시는 단순한 원형 객체인데도 bounding box 내부에는 객체가 아닌 배경 픽셀이 함께 들어갑니다. 실제 이미지에서는 객체 모양이 더 복잡하므로 이 차이는 더 커질 수 있습니다.


## 17-4. Segmentation label은 mask다

Segmentation에서는 정답을 bounding box 하나로 표현하지 않습니다. 대신 이미지와 같은 가로, 세로 크기를 가진 mask를 사용합니다.

가장 단순한 이진 segmentation에서는 다음처럼 표현할 수 있습니다.

- 배경 픽셀: `0`
- 객체 픽셀: `1`

여러 클래스를 구분하는 semantic segmentation에서는 픽셀마다 `0: background`, `1: road`, `2: person`, `3: car` 같은 클래스 id를 가질 수 있습니다.


In [ ]:
small_mask = np.array([
    [0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 0],
    [0, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0],
])

plt.figure(figsize=(4, 4))
plt.imshow(small_mask, cmap='gray', vmin=0, vmax=1)
plt.title('작은 binary mask 예시')
plt.xticks(range(small_mask.shape[1]))
plt.yticks(range(small_mask.shape[0]))
plt.grid(color='lightgray')
plt.show()

print(small_mask)


## 17-5. 어디에 쓰일까?

Segmentation은 위치를 정밀하게 알아야 하는 문제에서 중요합니다.

- 자율주행: 도로, 차선, 보행자, 차량 영역 구분
- 의료 영상: 종양, 장기, 병변 영역 추출
- 위성 영상: 건물, 도로, 산림, 물 영역 분류
- 배경 제거: 사람 또는 물체 영역만 분리
- 로봇 비전: 잡아야 할 물체의 정확한 영역 파악

공통점은 단순히 박스 위치만 필요한 것이 아니라, 픽셀 단위의 경계가 필요하다는 점입니다.


## 17-6. 정리

- Detection은 객체를 bounding box로 찾습니다.
- Segmentation은 객체 또는 클래스의 영역을 픽셀 단위 mask로 찾습니다.
- Bounding box는 단순하고 빠르지만 객체의 실제 모양을 정확히 표현하지 못합니다.
- Segmentation label은 이미지와 같은 공간 해상도를 가진 class map 또는 mask입니다.
- 다음 노트북에서는 semantic segmentation을 중심으로 픽셀 단위 classification 관점을 정리합니다.
